# AI Builders Club - NetworkEvolution | Session 1
## Script 03 : Ask the AI about your LIVE network  (script 02 + script 01)

**Goal:** combine what you built:
1. Show **sample questions** and wait for **your** question
2. First question -> get the live running config of **sw1 and sw2** (script 02) - **once**
3. Put both configs + your question into **one prompt**
4. Gemini answers using **your real config** (script 01)
5. Loop: ask again ... until you type `exit`

The AI has never seen your switches. It knows networking in general, but **not** your VLANs, IPs or usernames.
We fetch the facts ourselves and hand them over in the prompt -> the answer is about **your** network, not a textbook.

| Section | What you build | New? |
|---------|----------------|------|
| 1 | Imports (+ `genai`) + load `.env` | small change |
| 2 | Switch dictionaries | same as 02 |
| 3 | `get_running_config()` | same as 02 |
| 4 | `collect_configs()` -> one text for the prompt | **NEW** |
| 5 | Gemini client + model + `SYSTEM_INSTRUCTION` | **NEW** |
| 6 | Sample questions + `show_samples()` | **NEW** |
| 7 | `ask_ai(configs, question)` - one Gemini call | **NEW** |
| 8 | The question loop | **NEW** |


### How to use this notebook
Every **code cell** below is one section of the script `03_ask_ai_about_config.py`.

1. Create an empty file `03_ask_ai_about_config.py` in your session folder.
2. Copy the code cells **one by one, in order**, and paste each one at the **bottom** of that file.
3. After each paste, run it: `python 03_ask_ai_about_config.py` - and check the result described above the cell.

> Markdown cells (like this one) are the explanations - do **not** paste them into the `.py` file.

### Big picture: the "context in the prompt" pattern
```
  sw1 ---SSH--->  +--------------------+
                  |  Python script     |       +------------------+
  sw2 ---SSH--->  |  builds the PROMPT | ----> |  Gemini model    |
                  |                    |       |                  |
  you ---type-->  |  configs+question  | <---- |  answer based on |
                  +--------------------+       |  YOUR configs    |
                            |                  +------------------+
                            v
                      answer printed
```

**The prompt we send** (`input=...`):
```
=== DEVICE: sw1 ===
<live running config of sw1>
=== DEVICE: sw2 ===
<live running config of sw2>
QUESTION: Which VLANs are on sw1 but not on sw2?
```
**The system instruction** (`system_instruction=...`): role + rules - *answer ONLY from the configs, say which switch,
if it is not there say so, plain text, short*.

### Program flow
```
start -> show SAMPLE QUESTIONS
  |
  v
+--> "Your question:"  <------------------------------------+
|      +-- "exit"  -> stop                                  |
|      +-- "help"  -> show samples again -------------------+
|      +-- 1..7    -> pick that sample question             |
|      v                                                    |
|    configs collected already?                             |
|      NO  (1st question) -> SSH to sw1 + sw2 NOW (once)    |
|      YES (later)        -> reuse the SAME configs         |
|      v                                                    |
|    prompt = configs + question -> Gemini -> print answer -+
```
**Why collect configs only on the first question?** The script starts instantly, SSH happens once for many questions.
Downside: the configs are a **snapshot** - change a switch -> restart the script (script 04 adds `refresh`).

> **Note:** every question here is **independent** - the AI does **not** remember the previous one.
> *"and on sw2?"* will **not** work. That is script 04.

### Key words
| Word | Meaning |
|------|---------|
| **Prompt** | Everything the model reads: our data + the question |
| **Context** | The facts we add to the prompt (here: 2 configs) |
| **Grounding** | Making the model answer from **given facts** instead of its memory. The core idea behind RAG, AI agents and MCP servers |
| **Hallucination** | The model invents a confident but wrong answer. Grounding + "say you don't know" reduce it a lot (never to zero -> always verify) |
| **Context window** | Max tokens a model can read in one request. Gemini flash: ~1 million tokens -> 2 configs (~2,000 tokens) are tiny. 1,000 big routers would not fit -> that's when you need **RAG** |
| **Deterministic vs AI** | Python + SSH collect the data (always the same). The AI only **reads and explains** - it never logs in, never changes anything. A safe pattern to start with |

> **Security note:** we send the **full config** to a cloud API - including password hashes and SNMP communities.
> OK for a lab. In production: mask secrets **before** sending (regex "sanitizer"), check your company AI policy,
> or use a local model. (Covered in the full course.)

---
## Setup (one time - in the terminal, not in the `.py` file)

Skip this if you already did it for the earlier scripts.

```bash
pip install -r requirements.txt
cp .env.example .env
```
Your `.env` must contain your API key **and** the switch details:
```
GEMINI_API_KEY=paste-your-key-here
DEVICE_USERNAME=admin
DEVICE_PASSWORD=admin
SW1_HOST=172.20.20.11
SW2_HOST=172.20.20.12
# optional, default 22:
# SW1_PORT=22
# SW2_PORT=22
```
> `.env` holds secrets -> **never** share it, **never** commit it to git.

---
## Section 1 : Imports + load the `.env` file

Script 02 imports **plus** `from google import genai` (from script 01).

**Run it:** nothing printed - no error = all libraries installed.

In [ ]:
# ---------------------------------------------------------------
# SECTION 1 : Imports + load the .env file
# ---------------------------------------------------------------
import os

from dotenv import load_dotenv
from google import genai   # from script 01
from netmiko import ConnectHandler

load_dotenv()

---
## Section 2 : One dictionary per switch  (same as script 02)

All login details come from `.env`. Nothing new here.

In [ ]:
# ---------------------------------------------------------------
# SECTION 2 : One dictionary per switch  (same as 02)
# ---------------------------------------------------------------

USERNAME = os.getenv("DEVICE_USERNAME")
PASSWORD = os.getenv("DEVICE_PASSWORD")

SWITCHES = {
    "sw1": {
        "device_type": "arista_eos",
        "host": os.getenv("SW1_HOST"),
        "port": int(os.getenv("SW1_PORT", 22)),
        "username": USERNAME,
        "password": PASSWORD,
    },
    "sw2": {
        "device_type": "arista_eos",
        "host": os.getenv("SW2_HOST"),
        "port": int(os.getenv("SW2_PORT", 22)),
        "username": USERNAME,
        "password": PASSWORD,
    },
}

---
## Section 3 : SSH to one switch, return its config  (same as script 02)

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 : SSH to ONE switch and return its running config  (same as 02)
# ---------------------------------------------------------------

def get_running_config(device):
    with ConnectHandler(**device) as conn:
        conn.enable()
        return conn.send_command("show running-config")

---
## Section 4 : Collect BOTH configs into ONE text for the prompt  - NEW

Same loop as script 02, but instead of `print(config)` we **add** each config to a list, with a clear label in front:
```
=== DEVICE: sw1 ===
```
- The label tells the AI **which config belongs to which switch**.
- `parts.append(...)` -> add one item to the list
- `"\n\n".join(parts)` -> glue all pieces together with blank lines -> one big string
- A failed switch is skipped; if **both** fail the function returns `""` (empty text).

**Run it:** nothing printed yet - the function is only defined, called later in Section 8.

In [ ]:
# ---------------------------------------------------------------
# SECTION 4 : Collect BOTH configs into ONE text for the prompt
# ---------------------------------------------------------------

def collect_configs():
    parts = []
    for name, device in SWITCHES.items():
        print(f"Getting running config from {name} ({device['host']}) ...")
        try:
            config = get_running_config(device)
        except Exception as error:
            print(f"  {name}: FAILED -> {type(error).__name__}: {error}")
            continue

        print(f"  {name}: {len(config.splitlines())} lines")
        parts.append(f"=== DEVICE: {name} ===\n{config}")

    return "\n\n".join(parts)

---
## Section 5 : Gemini client + model + system instruction  - NEW

`SYSTEM_INSTRUCTION` = the **job description** of the AI. Every rule fixes a real problem:

| Rule | Problem it fixes |
|------|------------------|
| "Answer ONLY from the configs" | no generic textbook answers |
| "say: I don't see that in the config" | fewer hallucinations |
| "say which device each fact comes from" | we can verify the answer |
| "plain text, no markdown" | looks good in a terminal |
| "max 12 lines" | short, readable answers |

`""" ... """` = a multi-line string - easy to read and edit.

**Try it later:** remove the "ONLY from the configs" rule and ask *"Is BGP configured anywhere?"* - compare the answers.

In [ ]:
# ---------------------------------------------------------------
# SECTION 5 : Gemini client + model + system instruction
# ---------------------------------------------------------------
client = genai.Client()
MODEL = "gemini-3.8-flash"

SYSTEM_INSTRUCTION = """
You are a senior network engineer. You receive the LIVE running
configuration of Arista EOS switches, collected by a script a few
seconds ago. Each config starts with a line like "=== DEVICE: sw1 ===".

Rules:
- Answer ONLY from the configs you received. Do not guess.
- If the answer is not in the configs, say: "I don't see that in the config."
- Always say which device (sw1 / sw2) each fact comes from.
- Plain text for a terminal: short lines or "- " bullets.
  No markdown: no **bold**, no # headings, no tables.
- Keep it short (max 12 lines) unless the user asks for more detail.
"""

---
## Section 6 : Sample questions - printed as a separate block  - NEW

- A **list of strings** -> students type the **number** instead of the whole question (fast demo, no typos).
- `enumerate(list, start=1)` -> `(1, "..."), (2, "..."), ...`
- `"#" * 64` -> a line of 64 `#` = a clear visual separator

**Run it:** still nothing printed - `show_samples()` is called in Section 8.

In [ ]:
# ---------------------------------------------------------------
# SECTION 6 : Sample questions + show_samples()
# ---------------------------------------------------------------
SAMPLE_QUESTIONS = [
    "Which VLANs are configured on each switch? Show them with names.",
    "Compare sw1 and sw2. What is different?",
    "Which interfaces on sw1 have an IP address?",
    "Which users can log in and with which privilege level?",
    "Is any interface shutdown?",
    "Explain the sw1 config to a junior engineer in 5 lines.",
    "Is BGP configured anywhere?",
]


def show_samples():
    print("\n" + "#" * 64)
    print("#  SAMPLE QUESTIONS  - type a NUMBER or your own question")
    print("#" * 64)
    for number, question in enumerate(SAMPLE_QUESTIONS, start=1):
        print(f"   {number}. {question}")
    print("-" * 64)
    print("   help = show these questions again      exit = quit")
    print("#" * 64)

---
## Section 7 : ONE Gemini call - configs + question  - NEW

- `prompt = f"{configs_text}\n\nQUESTION: {question}"` -> an f-string glues data + question together
- Same `create()` call as script 01, plus our `SYSTEM_INSTRUCTION`
- We **return** the whole `interaction` -> the caller prints `output_text` and the tokens
- Every call is **independent** -> no memory of the previous question (memory = script 04)

In [ ]:
# ---------------------------------------------------------------
# SECTION 7 : ONE Gemini call - configs + question
# ---------------------------------------------------------------
def ask_ai(configs_text, question):
    prompt = f"{configs_text}\n\nQUESTION: {question}"
    return client.interactions.create(
        model=MODEL,
        input=prompt,
        system_instruction=SYSTEM_INSTRUCTION,
        generation_config={"thinking_level": "low"},
    )

---
## Section 8 : The question loop  - NEW (the program starts here)

| Code | Meaning |
|------|---------|
| `configs_text = None` | "not collected yet" |
| `while True:` | repeat until `exit` -> `break` |
| `input("...")` | waits for you to type, returns the text |
| `.strip()` | removes spaces / Enter around the text |
| `continue` | skip the rest, go back to "Your question:" |
| `question.isdigit()` | `"2"` -> pick sample question 2 (list index 1) |

- **First** question: `configs_text is None` -> SSH to both switches
- **Later** questions: reuse the **same** configs
- A failed AI call prints one line and the loop **keeps running**

**Run the full script now** and try:
| Type | What to observe |
|------|-----------------|
| `1` | VLANs per switch - from **your** config |
| `2` | sw1 vs sw2 differences |
| `7` | BGP is not configured -> *"I don't see that in the config"* (no hallucination) |
| `and on sw2?` | fails / confused - **no memory** -> script 04 |
| `exit` | quit |

Watch the **input tokens** (~2,000+): that's the two configs being sent **every** time.

In [ ]:
# ---------------------------------------------------------------
# SECTION 8 : The question loop
# ---------------------------------------------------------------
show_samples()
configs_text = None

while True:
    question = input("\nYour question: ").strip()

    # --- 8a : commands -------------------------------------------
    if not question:   # just Enter -> ask again
        continue
    if question.lower() in ("exit", "quit"):
        print("Bye!")
        break
    if question.lower() == "help":
        show_samples()
        continue
    if question.isdigit() and 1 <= int(question) <= len(SAMPLE_QUESTIONS):
        question = SAMPLE_QUESTIONS[int(question) - 1]          # "2" -> 2nd question (index 1)
        print(f"Your question: {question}")

    # --- 8b : first question -> collect the configs ONCE ---------
    if configs_text is None:
        print()
        configs_text = collect_configs()
        if not configs_text:                                    # both switches failed
            print("No config collected - check .env and the lab, then ask again.")
            configs_text = None                                 # try again next question
            continue
    else:
        print("(using the same configs collected for the first question)")

    # --- 8c : ask Gemini, print answer + tokens -------------------
    print("\nAsking Gemini ...\n")
    try:
        interaction = ask_ai(configs_text, question)
    except Exception as error:
        print(f"AI call failed -> {type(error).__name__} "
              f"(HTTP {getattr(error, 'status_code', '-')}): {str(error)[:300]}")
        continue                                                # keep the loop running

    print(f"AI: {interaction.output_text}")

    usage = interaction.usage
    print(f"\n[tokens -> input: {usage.total_input_tokens}, "
          f"output: {usage.total_output_tokens}, "
          f"thinking: {usage.total_thought_tokens}]")

---
## Recap - what you can now explain

- **Grounding:** the AI answers from **facts we give it** (live configs), not from its training memory.
- **Prompt = context + question.** The labels `=== DEVICE: sw1 ===` let the AI say *where* a fact comes from.
- **System instruction rules** shape the answer: only from configs, say "I don't see that", name the device.
- **Deterministic + AI:** Python/SSH collect the data, the AI only reads and explains - it never touches the switch.
- Every call is **independent** - no memory yet.
- **Security:** full configs go to a cloud API - in production, sanitize secrets first.

**Next:** Script 04 - turn this into a real **chat** that remembers the conversation.